# **Backtesting Combined Strategy**

**Importing previous code**

In [ ]:
import sys
import os
import numpy as np
import pandas as pd
import yfinance as yf
import datetime as dt
import matplotlib.pyplot as plt
from itertools import product

In [ ]:
class SMABackTester():      # Creating a class
  def __init__(self,company_symbol,period,interval,SMA_S,SMA_L):
    self.company_symbol=company_symbol
    self.period=period
    self.interval=interval
    self.SMA_S=SMA_S
    self.SMA_L=SMA_L
    self.result={}
    self.get_data()
    self.calculate_data()

  # Method: get_data()
  def get_data(self):
    stock=yf.download(tickers=self.company_symbol,period=self.period,interval= self.interval,auto_adjust=True)
    self.df=pd.DataFrame(stock["Close"])

  # Method: calculate_date():
  def calculate_data(self):
    for symbol in self.company_symbol:
      self.df[f"LR {symbol}"]=np.log(self.df[symbol]/self.df[symbol].shift(1))
      self.df[f"SMA_L {symbol}"]=self.df[symbol].rolling(window=self.SMA_L).mean()
      self.df[f"SMA_S {symbol}"]=self.df[symbol].rolling(window=self.SMA_S).mean()
      self.df.dropna(inplace=True)
      self.result[symbol] = self.df
    return self.df.dropna()

  # Method: set_parameters():Will be used for multiple combination of SMA_S & SMA_L
  def set_SMA_SnL(self,SMA_S=None,SMA_L=None):
    if self.SMA_S is not None:
      self.SMA_S=SMA_S
    if self.SMA_L is not None:
      self.SMA_L=SMA_L
    self.calculate_data()

  # Method: test_strategy(): will test the position and strategy for all SMA_S & SMA_L combination.
  def test_strategy(self):
    for symbol in self.company_symbol:
      self.df[f"Position {symbol}"]=np.where(self.df[f"SMA_S {symbol}"]>self.df[f"SMA_L {symbol}"],1,-1)
      self.df[f"Strategy {symbol}"]=self.df[f"Position {symbol}"].shift(1)*self.df[f"LR {symbol}"]
      self.df.dropna(inplace=True)
      self.df[f"Cumulative Return {symbol}"]=np.exp(self.df[f"LR {symbol}"].cumsum())
      self.df[f"Cumulative Strategy {symbol}"]=np.exp(self.df[f"Strategy {symbol}"].cumsum())
    return self.df.dropna()

  # Method: plot_result(): Will plot the results of the strategy for every ticker symbol
  def plot_result(self,ticker=None):
    if ticker is None:
      to_plot=self.company_symbol
    elif isinstance(ticker,str):
      to_plot=[ticker]
    else:
      to_plot=ticker
    for symbol in to_plot:
      if self.df[symbol] is None:
        print(f"Run the test_strategy() for {symbol} 1st!")
      else:
        fig,(ax1,ax2)=plt.subplots(2,1,figsize=(12,8),sharex=True)
        ax1.plot(self.df.index, self.df[symbol], label="Price", alpha=0.5)
        ax1.plot(self.df.index, self.df[f"SMA_S {symbol}"], label=f"SMA_S ({self.SMA_S})", color="orange")
        ax1.plot(self.df.index, self.df[f"SMA_L {symbol}"], label=f"SMA_L ({self.SMA_L})", color="blue")
        ax1.set_title(f"{symbol} - SMA Crossover")
        ax1.set_ylabel("Price")
        ax1.legend()
        ax2.plot(self.df.index, self.df[f"Cumulative Strategy {symbol}"], label="Strategy Performance", color="green")
        ax2.plot(self.df.index, self.df[f"Cumulative Return {symbol}"], label="Buy & Hold", color="gray", linestyle="--")
        ax2.set_ylabel("Growth ($1 Invested)")
        ax2.legend()
        plt.show()

  # Method: optimize_para(): make combination of SMA_S & SMA_L for different strategy output check
  def optimize_para(self,SMA_S_range,SMA_L_range):
    # making the combinations
    combination=list(product(range(*SMA_S_range),range(*SMA_L_range)))
    performance=[]
    for comb in combination:
      s,l=comb
      if s>l:
        continue
      self.set_parameters(s,l)
      # Access the result for the first company symbol and get the last 'Cumulative Strategy' value
      # Assuming optimization is based on the first company in the list
      df_first_company_result = self.test_strategy()[self.company_symbol[0]]
      performance.append(df_first_company_result["Cumulative Strategy"].iloc[-1])
    best_perf=np.max(performance)    # find best score
    opt=combination[np.argmax(performance)]    # find the SMA_S & SMA_L which gave the best score
    # Testing the best strategy by using optimal parameter (SMA_S & SMA_L)
    self.set_parameters(opt[0],opt[1])
    self.test_strategy()
    return self.performance

class BolingerBands():
  def __init__(self, company,period,interval):
    self.company=company
    self.period=period
    self.interval=interval
    self.get_data()

  def get_data(self):
    data=yf.download(self.company,period=self.period,interval=self.interval,auto_adjust=True)
    self.df=pd.DataFrame(data["Close"])
    return self.df

  def calculate_data(self):
    SMA=7;deviation=2;ptc=0.00007
    for ticker in self.company:
      self.df[f"LR {ticker}"]=np.log(self.df[ticker]/self.df[ticker].shift(1))
      self.df[f"SMA {ticker}"]=self.df[ticker].rolling(SMA).mean()
      # Corrected Bollinger Band calculation:
      self.df[f"Lower {ticker}"]=self.df[f"SMA {ticker}"] - (self.df[ticker].rolling(SMA).std()*deviation)
      self.df[f"Upper {ticker}"]=self.df[f"SMA {ticker}"] + (self.df[ticker].rolling(SMA).std()*deviation)
      self.df[f"Distance {ticker}"]=self.df[ticker]-self.df[f"SMA {ticker}"]
      conditions = [self.df[ticker] < self.df[f"Lower {ticker}"], # Buy if price is below the lower band
          self.df[ticker] > self.df[f"Upper {ticker}"] ] # Sell if price is above the upper band
      choices = [1, -1] # Corresponding choices for the conditions
      self.df[f"Position {ticker}"] = np.select(conditions, choices, default=0) # Default to 0 (neutral) if no conditions met

      self.df[f"Strategy {ticker}"]=self.df[f"Position {ticker}"].shift(1)*self.df[f"LR {ticker}"]
      self.df[f"Cumulative Return {ticker}"]=np.exp(self.df[f"LR {ticker}"].cumsum())
      self.df[f"Cumulative Strategy {ticker}"]=np.exp(self.df[f"Strategy {ticker}"].cumsum())
      self.df[f"Trades {ticker}"]=self.df[f"Position {ticker}"].diff().fillna(0).abs()
      self.df[f"Net_Strategy {ticker}"]=self.df[f"Strategy {ticker}"]-self.df[f"Trades {ticker}"]*ptc
      self.df[f"Cumulative Net_Strategy {ticker}"]=np.exp(self.df[f"Net_Strategy {ticker}"].cumsum())
    return self.df.dropna()

  def plot_data(self):
    for ticker in self.company:
      self.df[[f"{ticker}",f"SMA {ticker}",f"Lower {ticker}",f"Upper {ticker}"]].plot(figsize=(15,6),fontsize=12)
      plt.legend(fontsize=12)
      plt.show()
      self.df[[f"Cumulative Return {ticker}",f"Cumulative Strategy {ticker}"]].plot(figsize=(15,6),fontsize=12,title=f"Cumulative Result of {ticker}")
      plt.legend(fontsize=12)
      plt.show()
      self.df[[f"Cumulative Strategy {ticker}",f"Cumulative Net_Strategy {ticker}"]].plot(figsize=(15,6),fontsize=12,title=f"Cumulative Result of {ticker}")
      plt.legend(fontsize=12)
      plt.show()

  def ann_result(self):
    ann_return=[];ann_risk=[]
    for ticker in self.company:
      ann_return.append(self.df[[f"LR {ticker}",f"Net_Strategy {ticker}"]].mean()*(4*252))
      ann_risk.append(self.df[[f"LR {ticker}",f"Net_Strategy {ticker}"]].std()*np.sqrt(4*252))
    print(f"Annual Returns of companies = {ann_return}")
    print(f"Annual Risks of companies = {ann_risk}")

In [ ]:
symbol=["HAL.NS"];period="8d";timeframe="1m"
data1=SMABackTester(company_symbol=symbol,period=period,interval=timeframe,SMA_L=30,SMA_S=7)
data1.calculate_data();data1.test_strategy()

In [ ]:
data2=BolingerBands(company=symbol,period=period,interval=timeframe)
data2.calculate_data()

# **Combining Both Strategy:**

**Strategy - 1:** If both strategy returns same signal then final siganl will be one of them otherwise neutral

In [ ]:
comb=data1.df.loc[:,["LR HAL.NS","Position HAL.NS"]]
comb.rename(columns={"Position HAL.NS":"Position SMA"},inplace=True)
comb["Position BB"]=data2.df['Position HAL.NS']
comb["Position Combine"]=np.where(comb['Position BB']==comb['Position SMA'],comb['Position BB'],0)
comb["IST"]=comb.index.tz_convert("Asia/Kolkata")
comb["Hour"]=comb["IST"].dt.hour
comb['Position Combine']=np.where(comb['Hour'].between(10,14),comb["Position Combine"],0)   # 10 to 14 is busy trading hour (more values are in these hours)
comb['Position Combine'].value_counts()

**Starting Backtest**

In [ ]:
comb["Strategy"]=comb['Position Combine'].shift(1)*comb['LR HAL.NS']
comb["Trades"]=comb['Position Combine'].diff().abs()
trading_cost=0.0004
comb["Strategy"]=comb['Strategy']-comb["Trades"]*trading_cost
comb

In [ ]:
comb["Cumulative Returns"]=np.exp(comb['LR HAL.NS'].cumsum())
comb["Cumulative Strategy"]=np.exp(comb['Strategy'].cumsum())
comb[["Cumulative Returns","Cumulative Strategy"]].plot(figsize=(12,8),fontsize=12)
plt.xlabel("Time",fontsize=12);plt.ylabel("Returns",fontsize=12)
plt.title("HAL.NS Cumlative Returns & Cumulative Strategy")
plt.legend(fontsize=12);plt.show()

**Stretegy - 2:**
1. If the sign of $\text{signal(strategy1 + strategy 2)}$ is +ve $→$ final signal = +1 (buy)
2. If the sign of $\text{signal(strategy 1 + strategy 2)}$ is -ve $→$ final signal = -1 (sell)
3. If the sign of $ \text{signal(strategy 1 + strategy 2)}$ is 0 $→$ final signal = 0 (neutral)

In [ ]:
comb["Position Combine"]=np.sign(comb['Position BB']+comb['Position SMA'])
comb['Position Combine']=np.where(comb['Hour'].between(10,14),comb["Position Combine"],0)   # 10 to 14 is busy trading hour (more values are in these hours)
comb['Position Combine'].value_counts()

Backtesting the strategy 2:

In [ ]:
comb["Strategy"]=comb['Position Combine'].shift(1)*comb['LR HAL.NS']
comb["Trades"]=comb['Position Combine'].diff().abs()
trading_cost=0.0004
comb["Strategy"]=comb['Strategy']-comb["Trades"]*trading_cost
comb["Cumulative Returns"]=np.exp(comb['LR HAL.NS'].cumsum())
comb["Cumulative Strategy"]=np.exp(comb['Strategy'].cumsum())
comb[["Cumulative Returns","Cumulative Strategy"]].plot(figsize=(12,8),fontsize=12)
plt.xlabel("Time",fontsize=12);plt.ylabel("Returns",fontsize=12)
plt.title("HAL.NS Cumlative Returns & Cumulative Strategy")
plt.legend(fontsize=12);plt.show()